# Первичный обзор данных Olist

Цель ноутбука — проверить состав исходных данных, изучить размер и структуру рабочих таблиц, оценить типы данных, пропуски, дубликаты и базовые связи между таблицами.

На этом этапе данные не очищаются и не преобразуются, проводится только первичная диагностика, чтобы подготовить проект к созданию аналитической базы.


In [1]:
from pathlib import Path

import pandas as pd

In [2]:
RAW_DATA_DIR = Path("../data/raw")

project_files = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_products_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "product_category_name_translation.csv",
]

unused_files = [
    "olist_sellers_dataset.csv",
    "olist_geolocation_dataset.csv",
]

In [3]:
missing_files = []

for file_name in project_files:
    file_path = RAW_DATA_DIR / file_name

    if file_path.exists():
        print(f"Файл найден: {file_name}")
    else:
        print(f"Файл отсутствует: {file_name}")
        missing_files.append(file_name)

if missing_files:
    raise FileNotFoundError(f"Отсутствуют файлы: {missing_files}")

Файл найден: olist_orders_dataset.csv
Файл найден: olist_order_items_dataset.csv
Файл найден: olist_order_payments_dataset.csv
Файл найден: olist_customers_dataset.csv
Файл найден: olist_products_dataset.csv
Файл найден: olist_order_reviews_dataset.csv
Файл найден: product_category_name_translation.csv


In [4]:
tables = {}

for file_name in project_files:
    table_name = file_name.replace(".csv", "")
    file_path = RAW_DATA_DIR / file_name

    tables[table_name] = pd.read_csv(file_path)

In [5]:
list(tables.keys())

['olist_orders_dataset',
 'olist_order_items_dataset',
 'olist_order_payments_dataset',
 'olist_customers_dataset',
 'olist_products_dataset',
 'olist_order_reviews_dataset',
 'product_category_name_translation']

In [6]:
tables_overview = []

for table_name, df in tables.items():
    tables_overview.append(
        {
            "table_name": table_name,
            "rows": df.shape[0],
            "columns": df.shape[1],
        }
    )

tables_overview_df = (
    pd.DataFrame(tables_overview).sort_values(by="rows", ascending=False).reset_index(drop=True)
)

tables_overview_df

,table_name,rows,columns
0,olist_order_items_dataset,112650,7
1,olist_order_payments_dataset,103886,5
2,olist_orders_dataset,99441,8
3,olist_customers_dataset,99441,5
4,olist_order_reviews_dataset,99224,7
5,olist_products_dataset,32951,9
6,product_category_name_translation,71,2


In [7]:
for table_name, df in tables.items():
    print(f"\n{table_name}")
    print(df.columns.tolist())


olist_orders_dataset
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

olist_order_items_dataset
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

olist_order_payments_dataset
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

olist_customers_dataset
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

olist_products_dataset
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

olist_order_reviews_dataset
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

product_cate

In [8]:
dtypes_overview = []

for table_name, df in tables.items():
    for column_name, dtype in df.dtypes.items():
        dtypes_overview.append(
            {
                "table_name": table_name,
                "column_name": column_name,
                "dtype": str(dtype),
            }
        )

dtypes_overview_df = pd.DataFrame(dtypes_overview)

dtypes_overview_df

,table_name,column_name,dtype
0,olist_orders_dataset,order_id,object
1,olist_orders_dataset,customer_id,object
2,olist_orders_dataset,order_status,object
3,olist_orders_dataset,order_purchase_timestamp,object
4,olist_orders_dataset,order_approved_at,object
5,olist_orders_dataset,order_delivered_carrier_date,object
6,olist_orders_dataset,order_delivered_customer_date,object
7,olist_orders_dataset,order_estimated_delivery_date,object
8,olist_order_items_dataset,order_id,object
9,olist_order_items_dataset,order_item_id,int64


In [9]:
missing_values_overview = []

for table_name, df in tables.items():
    for column_name in df.columns:
        missing_count = df[column_name].isna().sum()
        missing_share_percent = round(missing_count / len(df) * 100, 2)

        missing_values_overview.append(
            {
                "table_name": table_name,
                "column_name": column_name,
                "missing_count": missing_count,
                "missing_share_percent": missing_share_percent,
            }
        )

missing_values_overview_df = (
    pd.DataFrame(missing_values_overview)
    .sort_values(by="missing_count", ascending=False)
    .reset_index(drop=True)
)

missing_values_overview_df

,table_name,column_name,missing_count,missing_share_percent
0,olist_order_reviews_dataset,review_comment_title,87656,88.34
1,olist_order_reviews_dataset,review_comment_message,58247,58.70
2,olist_orders_dataset,order_delivered_customer_date,2965,2.98
3,olist_orders_dataset,order_delivered_carrier_date,1783,1.79
4,olist_products_dataset,product_category_name,610,1.85
5,olist_products_dataset,product_name_lenght,610,1.85
6,olist_products_dataset,product_description_lenght,610,1.85
7,olist_products_dataset,product_photos_qty,610,1.85
8,olist_orders_dataset,order_approved_at,160,0.16
9,olist_products_dataset,product_length_cm,2,0.01


In [10]:
missing_values_overview_df[missing_values_overview_df["missing_count"] > 0]

,table_name,column_name,missing_count,missing_share_percent
0,olist_order_reviews_dataset,review_comment_title,87656,88.34
1,olist_order_reviews_dataset,review_comment_message,58247,58.70
2,olist_orders_dataset,order_delivered_customer_date,2965,2.98
3,olist_orders_dataset,order_delivered_carrier_date,1783,1.79
4,olist_products_dataset,product_category_name,610,1.85
5,olist_products_dataset,product_name_lenght,610,1.85
6,olist_products_dataset,product_description_lenght,610,1.85
7,olist_products_dataset,product_photos_qty,610,1.85
8,olist_orders_dataset,order_approved_at,160,0.16
9,olist_products_dataset,product_length_cm,2,0.01


## Первичный просмотр данных

Быстрый просмотр первых строк рабочих таблиц для понимания фактического содержания полей.


In [11]:
for table_name, df in tables.items():
    print(f"\n{table_name}")
    display(df.head())


olist_orders_dataset


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



olist_order_items_dataset


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14



olist_order_payments_dataset


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0



olist_order_reviews_dataset


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53



product_category_name_translation


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


## Проверка дубликатов и уникальности ключей

Проверяется наличие полных дубликатов строк и корректность основных идентификаторов перед будущими объединениями таблиц.


In [12]:
duplicates_overview = []

for table_name, df in tables.items():
    duplicate_rows = df.duplicated().sum()
    duplicate_share_percent = round(df.duplicated().mean() * 100, 2)

    duplicates_overview.append(
        {
            "table_name": table_name,
            "rows": len(df),
            "duplicate_rows": duplicate_rows,
            "duplicate_share_percent": duplicate_share_percent,
        }
    )

duplicates_overview_df = pd.DataFrame(duplicates_overview)

duplicates_overview_df

,table_name,rows,duplicate_rows,duplicate_share_percent
0,olist_orders_dataset,99441,0,0.0
1,olist_order_items_dataset,112650,0,0.0
2,olist_order_payments_dataset,103886,0,0.0
3,olist_customers_dataset,99441,0,0.0
4,olist_products_dataset,32951,0,0.0
5,olist_order_reviews_dataset,99224,0,0.0
6,product_category_name_translation,71,0,0.0


In [13]:
key_columns = {
    "olist_orders_dataset": "order_id",
    "olist_customers_dataset": "customer_id",
    "olist_products_dataset": "product_id",
    "olist_order_reviews_dataset": "review_id",
}

key_uniqueness_overview = []

for table_name, key_column in key_columns.items():
    df = tables[table_name]
    duplicated_keys = df[key_column].duplicated().sum()

    key_uniqueness_overview.append(
        {
            "table_name": table_name,
            "key_column": key_column,
            "rows": len(df),
            "unique_keys": df[key_column].nunique(),
            "duplicated_keys": duplicated_keys,
            "duplicated_keys_share_percent": round(duplicated_keys / len(df) * 100, 2),
        }
    )

key_uniqueness_overview_df = pd.DataFrame(key_uniqueness_overview)

key_uniqueness_overview_df

,table_name,key_column,rows,unique_keys,duplicated_keys,duplicated_keys_share_percent
0,olist_orders_dataset,order_id,99441,99441,0,0.00
1,olist_customers_dataset,customer_id,99441,99441,0,0.00
2,olist_products_dataset,product_id,32951,32951,0,0.00
3,olist_order_reviews_dataset,review_id,99224,98410,814,0.82


In [14]:
orders = tables["olist_orders_dataset"]

order_status_overview = orders["order_status"].value_counts().reset_index()

order_status_overview.columns = ["order_status", "orders_count"]

order_status_overview["orders_share_percent"] = (
    order_status_overview["orders_count"] / len(orders) * 100
).round(2)

order_status_overview

,order_status,orders_count,orders_share_percent
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [15]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders_dates = orders.copy()

for column in date_columns:
    orders_dates[column] = pd.to_datetime(orders_dates[column], errors="coerce")

date_range_overview = []

for column in date_columns:
    missing_count = orders_dates[column].isna().sum()

    date_range_overview.append(
        {
            "column_name": column,
            "min_date": orders_dates[column].min(),
            "max_date": orders_dates[column].max(),
            "missing_count": missing_count,
            "missing_share_percent": round(missing_count / len(orders_dates) * 100, 2),
        }
    )

date_range_overview_df = pd.DataFrame(date_range_overview)

date_range_overview_df

,column_name,min_date,max_date,missing_count,missing_share_percent
0,order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18,0,0.00
1,order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,160,0.16
2,order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,1783,1.79
3,order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,2965,2.98
4,order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0,0.00


In [16]:
orders = tables["olist_orders_dataset"]
order_items = tables["olist_order_items_dataset"]
payments = tables["olist_order_payments_dataset"]
reviews = tables["olist_order_reviews_dataset"]

orders_ids = set(orders["order_id"])

related_tables = {
    "olist_order_items_dataset": order_items,
    "olist_order_payments_dataset": payments,
    "olist_order_reviews_dataset": reviews,
}

order_id_coverage = []

for table_name, df in related_tables.items():
    related_order_ids = set(df["order_id"])

    order_id_coverage.append(
        {
            "table_name": table_name,
            "rows": len(df),
            "unique_order_ids": df["order_id"].nunique(),
            "orders_without_match_in_orders": len(related_order_ids - orders_ids),
            "orders_without_match_in_related_table": len(orders_ids - related_order_ids),
            "orders_coverage_percent": round(
                df["order_id"].nunique() / orders["order_id"].nunique() * 100,
                2,
            ),
        }
    )

order_id_coverage_df = pd.DataFrame(order_id_coverage)

order_id_coverage_df

,table_name,rows,unique_order_ids,orders_without_match_in_orders,orders_without_match_in_related_table,orders_coverage_percent
0,olist_order_items_dataset,112650,98666,0,775,99.22
1,olist_order_payments_dataset,103886,99440,0,1,100.00
2,olist_order_reviews_dataset,99224,98673,0,768,99.23


In [17]:
customers = tables["olist_customers_dataset"]

customers_overview = {
    "rows": len(customers),
    "unique_customer_id": customers["customer_id"].nunique(),
    "unique_customer_unique_id": customers["customer_unique_id"].nunique(),
    "duplicated_customer_id": customers["customer_id"].duplicated().sum(),
    "customers_with_multiple_customer_id": (
        customers.groupby("customer_unique_id")["customer_id"].nunique().gt(1).sum()
    ),
}

customers_overview

{'rows': 99441,
 'unique_customer_id': 99441,
 'unique_customer_unique_id': 96096,
 'duplicated_customer_id': np.int64(0),
 'customers_with_multiple_customer_id': np.int64(2997)}

In [18]:
products = tables["olist_products_dataset"]
category_translation = tables["product_category_name_translation"]

product_categories_overview = {
    "products_rows": len(products),
    "unique_product_id": products["product_id"].nunique(),
    "products_without_category": products["product_category_name"].isna().sum(),
    "unique_categories_in_products": products["product_category_name"].nunique(),
    "unique_categories_in_translation": category_translation["product_category_name"].nunique(),
}

product_categories_overview

{'products_rows': 32951,
 'unique_product_id': 32951,
 'products_without_category': np.int64(610),
 'unique_categories_in_products': 73,
 'unique_categories_in_translation': 71}

In [19]:
product_categories = set(products["product_category_name"].dropna())
translated_categories = set(category_translation["product_category_name"])

category_translation_coverage = {
    "categories_without_translation_count": len(product_categories - translated_categories),
    "categories_without_translation": sorted(product_categories - translated_categories),
    "translations_without_products_count": len(translated_categories - product_categories),
    "translations_without_products": sorted(translated_categories - product_categories),
}

category_translation_coverage

{'categories_without_translation_count': 2,
 'categories_without_translation': ['pc_gamer',
  'portateis_cozinha_e_preparadores_de_alimentos'],
 'translations_without_products_count': 0,
 'translations_without_products': []}

In [20]:
payments = tables["olist_order_payments_dataset"]

payments_overview = {
    "rows": len(payments),
    "unique_order_ids": payments["order_id"].nunique(),
    "orders_with_multiple_payment_rows": (payments.groupby("order_id").size().gt(1).sum()),
    "min_payment_value": payments["payment_value"].min(),
    "max_payment_value": payments["payment_value"].max(),
    "mean_payment_value": payments["payment_value"].mean().round(2),
}

payments_overview

{'rows': 103886,
 'unique_order_ids': 99440,
 'orders_with_multiple_payment_rows': np.int64(2961),
 'min_payment_value': np.float64(0.0),
 'max_payment_value': np.float64(13664.08),
 'mean_payment_value': np.float64(154.1)}

In [21]:
reviews = tables["olist_order_reviews_dataset"]

reviews_overview = {
    "rows": len(reviews),
    "unique_review_ids": reviews["review_id"].nunique(),
    "unique_order_ids": reviews["order_id"].nunique(),
    "orders_with_multiple_reviews": (reviews.groupby("order_id").size().gt(1).sum()),
    "min_review_score": reviews["review_score"].min(),
    "max_review_score": reviews["review_score"].max(),
    "mean_review_score": reviews["review_score"].mean().round(2),
}

reviews_overview

{'rows': 99224,
 'unique_review_ids': 98410,
 'unique_order_ids': 98673,
 'orders_with_multiple_reviews': np.int64(547),
 'min_review_score': np.int64(1),
 'max_review_score': np.int64(5),
 'mean_review_score': np.float64(4.09)}

In [22]:
review_score_distribution = reviews["review_score"].value_counts().sort_index().reset_index()

review_score_distribution.columns = ["review_score", "reviews_count"]

review_score_distribution["reviews_share_percent"] = (
    review_score_distribution["reviews_count"] / len(reviews) * 100
).round(2)

review_score_distribution

,review_score,reviews_count,reviews_share_percent
0,1,11424,11.51
1,2,3151,3.18
2,3,8179,8.24
3,4,19142,19.29
4,5,57328,57.78


## Первичные наблюдения

По результатам первичного обзора данных:

- В активный анализ включены 7 таблиц: заказы, товары в заказах, оплаты, клиенты, товары, отзывы и перевод категорий.
- Таблицы `olist_geolocation_dataset.csv` и `olist_sellers_dataset.csv` на текущем этапе не используются, так как они не связаны напрямую с выбранными бизнес-вопросами.
- Основной таблицей для анализа заказов является `olist_orders_dataset.csv`.
- Для анализа выручки и товарных категорий потребуется объединение заказов, товаров в заказах, товаров и перевода категорий.
- Для части товарных категорий может отсутствовать перевод, поэтому перед анализом категорий нужно отдельно обработать категории без английского названия.
- Для анализа новых и повторных клиентов нужно использовать `customer_unique_id`, а не `customer_id`.
- Даты заказов загружаются как строковые значения и позже должны быть преобразованы в формат `datetime`.
- Для основной аналитики нужно отдельно определить, какие статусы заказов включать в расчёт метрик.
- Таблицы оплат и отзывов требуют осторожного объединения, так как на один заказ может приходиться несколько строк; перед объединением их может потребоваться агрегировать до уровня заказа.
